# 05 - Ablations, sensitivity and robustness sweeps (P9)

Training cells resume from `DONE` markers, so re-running after a disconnect continues. Run matrices are defined in `configs/ablations.yaml`; every run uses the quad, arm A3, 3 seeds and the full-model budget.

In [ ]:
!pip -q install stable-baselines3==2.9.0 scipy pandas matplotlib pyyaml

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)

In [ ]:
import copy, yaml
from training.algos import load_training_config
from training.runner import train
cfg = load_training_config()
BUDGET = int(cfg['budgets']['guidance_train_steps'])
ABL = yaml.safe_load((ROOT / 'configs' / 'ablations.yaml').read_text())
SEEDS = cfg['seeds']['training']
BUDGET

In [ ]:
def base(seed):
    return {'task': 'guidance', 'algo': 'ppo', 'seed': seed, 'total_steps': BUDGET,
            'checkpoint_every': 25000, 'asymmetric': True, 'actor_obs_dim': 85,
            'guidance': {'platform': 'quadcopter', 'mode': 'apn_rl', 'target_set': 'evasive'}}

def merge(a, b):
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out

def run_group(group, prefix):
    for name, delta in ABL[group].items():
        for seed in SEEDS:
            train(f'{prefix}_{name}_s{seed}', merge(base(seed), delta))

## P9.2 ablations (no privileged critic, no history block, no prediction block)

In [ ]:
run_group('guidance_ablations', 'p9a')

## P9.3 design-choice sensitivity (residual authority, discount)

In [ ]:
run_group('sensitivity', 'p9s')

## Reward-weight sweep (kill bonus, progress, residual penalty at 0.5x and 1.5x)

In [ ]:
run_group('reward_weights', 'p9r')

## P9.5 flight-controller domain randomisation

In [ ]:
for name, delta in ABL['fc_randomisation'].items():
    for seed in SEEDS:
        train(f'p9f_{name}_s{seed}', {'task': 'fc', 'algo': 'ppo', 'seed': seed,
              'total_steps': int(cfg['budgets']['fc_train_steps']), 'checkpoint_every': 25000,
              'asymmetric': True, **delta})

## P9.1 / P9.6 sweeps and P9.4 residual analysis (evaluation only)

In [ ]:
!python scripts/sweep_robustness.py --episodes 5 --workers 2 --policy quadcopter=runs/p7_quadcopter_a3_s0/checkpoint.zip --policy fixed_wing=runs/p7_fixed_wing_a3_s0/checkpoint.zip --policy missile=runs/p7_missile_a3_s0/checkpoint.zip
!python scripts/residual_analysis.py --policy quadcopter=runs/p7_quadcopter_a3_s0/checkpoint.zip --policy fixed_wing=runs/p7_fixed_wing_a3_s0/checkpoint.zip --policy missile=runs/p7_missile_a3_s0/checkpoint.zip
!python scripts/sweep_bridge_rate.py --episodes 5 --workers 2 --guidance-policy runs/p7_quadcopter_a3_s0/checkpoint.zip

Evaluate each ablation checkpoint with `scripts/evaluate_guidance_arms.py --policy a3_quadcopter=<run>/checkpoint.zip` and tabulate kill rate and median miss against the full model on the common evaluation seeds.